# LEGO Brick Color Detector

Run every cell from top to bottom (Runtime → Run all), then upload a photo of LEGO bricks when prompted at the bottom.
The model will draw bounding boxes around each brick and print a count per color.

No Google account / Drive access needed — the model is pulled straight from this public GitHub repo.


In [ ]:
# Setup (run once per Colab session)
!pip install -q ultralytics

# Clone the repo to get the trained model (skips re-cloning if already present)
import os
if not os.path.exists('lego-detector'):
    !git clone --depth 1 https://github.com/otho560/lego-detector.git
%cd lego-detector


In [ ]:
# Load model + define the count function (run once per session)

from ultralytics import YOLO
from collections import Counter
import matplotlib.pyplot as plt

MODEL_PATH = 'models/lego_model.pt'
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

def count_legos(image_path, conf=0.5):
    results = model(image_path, conf=conf, verbose=False)
    r = results[0]
    counts = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    total = sum(counts.values())

    plt.figure(figsize=(12, 12))
    plt.imshow(r.plot()[..., ::-1])
    plt.axis('off')
    title = f"Total: {total}  |  " + "  ".join(f"{k}: {v}" for k, v in sorted(counts.items()))
    plt.title(title, fontsize=13)
    plt.show()

    print(f"\n=== LEGO Count ===")
    print(f"Total bricks: {total}")
    for color, n in sorted(counts.items()):
        print(f"  {color:8s}: {n}")
    return dict(counts)


In [ ]:
# Upload an image and predict (run every time you want to test a new pic)

from google.colab import files

uploaded = files.upload()  # opens a file picker
for filename in uploaded.keys():
    print(f"\n--- {filename} ---")
    count_legos(filename)
